# Exercício 2: Bronze de Eventos de Acesso (Schema Evolution)

Notebook de exploração da tabela Delta gerada pelo pipeline `src/lakehouse_bronze_eventos_acesso.py`.

Tabela: `data/bronze/bronze_eventos_acesso` (3 lotes consolidados via append + `mergeSchema`: dia 1 CSV sem `dispositivo`/`metadata`, dia 2 CSV com `dispositivo`, dia 3 JSON com `metadata` aninhado).

In [1]:
import sys

# permite importar os módulos de src/ quando o notebook roda a partir de notebooks/
sys.path.insert(0, "../src")

from lakehouse_bronze_eventos_acesso import read_bronze_eventos_com_app_version
from lakehouse_bronze_matriculas import create_spark_session

spark = create_spark_session(app_name="notebook_exercicio_2")
bronze_path = "../data/bronze/bronze_eventos_acesso"

:: loading settings :: url = jar:file:/Users/jpdagostin/Desktop/personal/lakehouse_education/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/jpdagostin/.ivy2.5.2/cache
The jars for the packages stored in: /Users/jpdagostin/.ivy2.5.2/jars
io.delta#delta-spark_4.1_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-234c04e8-9cfe-4d9c-af3d-ebf4ee7da806;1.0
	confs: [default]
	found io.delta#delta-spark_4.1_2.13;4.3.1 in central
	found io.delta#delta-storage;4.3.1 in central
	found io.unitycatalog#unitycatalog-client;0.5.0 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	found org.apache.logging.log4j#log4j-slf4j2-impl;2.25.3 in central
	found org.apache.logging.log4j#log4j-api;2.25.3 in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found io.unitycatalog#unitycatalog-hadoop;0.5.0 in central


	found org.apache.logging.log4j#log4j-core;2.25.3 in central
	found io.delta#delta-kernel-api;4.3.1 in central
	found org.roaringbitmap#RoaringBitmap;0.9.25 in central
	found com.fasterxml.jackson.core#jackson-databind;2.13.5 in central
	found com.fasterxml.jackson.core#jackson-annotations;2.13.5 in central
	found com.fasterxml.jackson.core#jackson-core;2.13.5 in central
	found com.fasterxml.jackson.datatype#jackson-datatype-jdk8;2.13.5 in central
	found org.roaringbitmap#shims;0.9.25 in central
	found io.delta#delta-kernel-defaults;4.3.1 in central
	found org.apache.hadoop#hadoop-client-runtime;3.4.2 in central
	found org.apache.parquet#parquet-hadoop;1.16.0 in central
	found org.apache.parquet#parquet-column;1.16.0 in central
	found org.apache.parquet#parquet-common;1.16.0 in central
	found org.apache.parquet#parquet-format-structures;1.16.0 in central
	found javax.annotation#javax.annotation-api;1.3.2 in central
	found org.apache.parquet#parquet-encoding;1.16.0 in central
	found org

26/08/08 20:57:53 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


## Schema consolidado (evoluído a partir dos 3 lotes)

In [2]:
df_bronze = spark.read.format("delta").load(bronze_path)
df_bronze.printSchema()

root
 |-- evento_id: string (nullable = true)
 |-- aluno_id: string (nullable = true)
 |-- tipo_evento: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- dispositivo: string (nullable = true)
 |-- metadata: struct (nullable = true)
 |    |-- app_version: string (nullable = true)
 |    |-- os: string (nullable = true)



## Todos os eventos, ordenados por timestamp

Repare que `dispositivo` é `null` para os registros do dia 1, e `metadata` é `null` para os registros dos dias 1 e 2 (colunas que ainda não existiam quando esses lotes foram gravados).

In [3]:
df_bronze.orderBy("timestamp").show(truncate=False)

26/08/08 20:57:58 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+---------+--------+-------------------+-------------------+-----------+----------------+
|evento_id|aluno_id|tipo_evento        |timestamp          |dispositivo|metadata        |
+---------+--------+-------------------+-------------------+-----------+----------------+
|e1001    |2001    |login              |2026-02-01 08:00:00|NULL       |NULL            |
|e1002    |2001    |acesso_videoaula   |2026-02-01 08:05:00|NULL       |NULL            |
|e1003    |2002    |submissao_exercicio|2026-02-01 09:20:00|NULL       |NULL            |
|e2001    |2001    |login              |2026-02-02 08:00:00|mobile     |NULL            |
|e2002    |2003    |login              |2026-02-02 08:30:00|web        |NULL            |
|e2003    |2003    |acesso_videoaula   |2026-02-02 08:45:00|web        |NULL            |
|e3001    |2001    |login              |2026-02-03 08:00:00|mobile     |{4.2.1, android}|
|e3002    |2002    |submissao_exercicio|2026-02-03 09:00:00|web        |NULL            |
+---------

## Consulta tolerante a `metadata` ausente

Acessar `metadata.app_version` em uma linha onde a coluna inteira é `null` (dias 1 e 2) retorna `null` naturalmente, sem erro.

In [4]:
read_bronze_eventos_com_app_version(spark, bronze_path).orderBy("timestamp").show(truncate=False)

+---------+--------+-------------------+-------------------+-----------+-----------+
|evento_id|aluno_id|tipo_evento        |timestamp          |dispositivo|app_version|
+---------+--------+-------------------+-------------------+-----------+-----------+
|e1001    |2001    |login              |2026-02-01 08:00:00|NULL       |NULL       |
|e1002    |2001    |acesso_videoaula   |2026-02-01 08:05:00|NULL       |NULL       |
|e1003    |2002    |submissao_exercicio|2026-02-01 09:20:00|NULL       |NULL       |
|e2001    |2001    |login              |2026-02-02 08:00:00|mobile     |NULL       |
|e2002    |2003    |login              |2026-02-02 08:30:00|web        |NULL       |
|e2003    |2003    |acesso_videoaula   |2026-02-02 08:45:00|web        |NULL       |
|e3001    |2001    |login              |2026-02-03 08:00:00|mobile     |4.2.1      |
|e3002    |2002    |submissao_exercicio|2026-02-03 09:00:00|web        |NULL       |
+---------+--------+-------------------+-------------------+-----

## Histórico de versões (uma por lote/append)

In [5]:
from delta.tables import DeltaTable

DeltaTable.forPath(spark, bronze_path).history().select(
    "version", "timestamp", "operation", "operationParameters"
).show(truncate=False)

+-------+-----------------------+---------+-----------------------------------------------------------+
|version|timestamp              |operation|operationParameters                                        |
+-------+-----------------------+---------+-----------------------------------------------------------+
|2      |2026-08-08 19:51:59.906|WRITE    |{mode -> Append, partitionBy -> [], canMergeSchema -> true}|
|1      |2026-08-08 19:51:59.58 |WRITE    |{mode -> Append, partitionBy -> [], canMergeSchema -> true}|
|0      |2026-08-08 19:51:57.166|WRITE    |{mode -> Append, partitionBy -> [], canMergeSchema -> true}|
+-------+-----------------------+---------+-----------------------------------------------------------+

